In [3]:
import torch
from rdkit import Chem
from rdkit.Chem import AllChem
import torch.nn as nn

In [23]:
ELEMENTS = ['C', 'N', 'O', 'S', 'F', 'Cl', 'Br', 'I', 'P', 'B']
DEGREES  = [0, 1, 2, 3, 4, 5]
CHARGES  = [-2, -1, 0, 1, 2]
NUM_HS   = [0, 1, 2, 3, 4]
HYBRIDS  = [Chem.HybridizationType.SP,  Chem.HybridizationType.SP2,
            Chem.HybridizationType.SP3, Chem.HybridizationType.SP3D,
            Chem.HybridizationType.SP3D2]
BONDTYPES = [Chem.BondType.SINGLE, Chem.BondType.DOUBLE,
             Chem.BondType.TRIPLE, Chem.BondType.AROMATIC]
STEREOS   = [Chem.BondStereo.STEREONONE, Chem.BondStereo.STEREOZ,
             Chem.BondStereo.STEREOE]

In [24]:
smiles, label = 'CC(C)=CCCC(C)=CC(=O)', -2.060
mol = Chem.MolFromSmiles(smiles)
atom = next(mol.GetAtoms())
bond = next(mol.GetBonds())

In [25]:

def one_hot(value, vocab):
    """One-hot with a trailing 'other' slot. Length = len(vocab) + 1."""
    v = [0.0] * (len(vocab) + 1)
    v[vocab.index(value) if value in vocab else len(vocab)] = 1.0
    return v

def get_atom_features(atom):
    f = one_hot(atom.GetSymbol(), ELEMENTS)
    f += one_hot(atom.GetDegree(), DEGREES)
    f += one_hot(atom.GetFormalCharge(), CHARGES)
    f += one_hot(atom.GetTotalNumHs(), NUM_HS)
    f += one_hot(atom.GetHybridization(), HYBRIDS)
    f += [float(atom.GetIsAromatic())]
    f += [float(atom.IsInRing())]
    q = atom.GetDoubleProp('_GasteigerCharge')
    f += [0.0 if (q != q or abs(q) > 1e3) else q]     #  1  (NaN/inf guard)
    return f

ATOM_DIM = 39

def get_bond_features(bond):
    f = one_hot(bond.GetBondType(), BONDTYPES)
    f += [float(bond.GetIsConjugated())]
    f += [float(bond.IsInRing())]
    f += one_hot(bond.GetStereo(), STEREOS)
    return f

BOND_DIM = 11

def mol_to_graph(smiles):
    """Return dict with X (N,ATOM_DIM), A (N,N), E (N,N,BOND_DIM), n."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError(f"RDkit cannot parse {smiles}")
    AllChem.ComputeGasteigerCharges(mol)
    n = mol.GetNumAtoms()
    X = torch.tensor([get_atom_features(a) for a in mol.GetAtoms()],
                     dtype=torch.float64)
    A = torch.zeros(n, n, dtype=torch.float64)
    E = torch.zeros(n, n, BOND_DIM, dtype=torch.float64)
    for b in mol.GetBonds():
        i, j = b.GetBeginAtomIdx(), b.GetEndAtomIdx()
        A[i, j] = A[j, i] = 1.0
        bf = torch.tensor(get_bond_features(b), dtype=torch.float64)
        E[i, j] = E[j, i] = bf
    return {'X': X, 'A': A, 'E': E, 'n': n, 'smiles': smiles}

graph = mol_to_graph(smiles)

def normalized_adjacency(A):
    A_hat = A + torch.eye(A.size(-1), dtype=A.dtype, device=A.device)
    deg = A.sum(dim=1)
    d_inv_sqrt = deg.clamp(min=1e-12).pow(-0.5)
    return d_inv_sqrt.unsqueeze(-1) * A_hat * d_inv_sqrt.unsqueeze(-2)

A_norm = normalized_adjacency(graph['A'])

def collate(graphs):
    B, Nmax = len(graphs), max(g['n'] for g in graphs)
    X = torch.zeros(B, Nmax, ATOM_DIM, dtype=torch.float64)
    A = torch.zeros(B, Nmax, Nmax, dtype=torch.float64)
    mask = torch.zeros(B, Nmax, dtype=torch.float64)

    for b, g in enumerate(graphs):
        n = g['n']
        X[b, :n] = g['X']
        A[b, :n, :n] = g['A']
        mask[b, :n] = 1.0

    return X, normalized_adjacency(A), mask

graphs = [graph, graph]
X, A_norm, mask = collate(graphs)

def gcn_forward(S, H, W, b):
    M = S @ H
    Z = (M @ W) + b
    Hout = torch.relu(Z)
    return Hout, (S, H, W, M, Z)

def gcn_backward(dHout, cache):
    """Hand-derived gradients of §3. dHout is dL/dHout, shape (B,N,F')."""
    S, H, W, M, Z = cache
    dZ = dHout * (Z > 0).to(Z.dtype)              # Delta = G * relu'(Z)
    dW = torch.einsum('bnf,bng->fg', M, dZ)       # (SH)^T Delta, summed over batch
    db = dZ.sum(dim=(0, 1))                       # column sum of Delta
    dM = dZ @ W.transpose(-1, -2)                 # Delta W^T
    dH = S.transpose(-1, -2) @ dM                 # S^T Delta W^T
    return dH, dW, db

class GCNRegressor(nn.Module):
    def __init__(self, in_dims=ATOM_DIM, hidden_dims=64, mlp_hidden=64):
        super().__init__()
        self.W1 = nn.Parameter(torch.empty(in_dims, hidden_dims, dtype=torch.float64))
        self.b1 = nn.Parameter(torch.zeros(hidden_dims, dtype=torch.float64))

        self.W2 = nn.Parameter(torch.empty(hidden_dims, hidden_dims, dtype=torch.float64))
        self.b2 = nn.Parameter(torch.zeros(hidden_dims, dtype=torch.float64))

        nn.init.xavier_uniform_(self.W1)
        nn.init.xavier_uniform_(self.W2)

        self.head = nn.Sequential(
            nn.Linear(hidden_dims, mlp_hidden, dtype=torch.float64),
            nn.ReLU(),
            nn.Linear(mlp_hidden, 1, dtype=torch.float64)
        )

    def forward(self, X, S, mask):
        H, _ = gcn_forward(S, X, self.W1, self.b1)
        H, _ = gcn_forward(S, H, self.W2, self.b2)
        H = H * mask.unsqueeze(-1)
        z = H.sum(1)/mask.sum(1, keepdim=True)
        return self.head(z).squeeze(-1)
    
X, S, mask = collate(graphs)
model = GCNRegressor()
model(X, S, mask)

tensor([0.0411, 0.0411], dtype=torch.float64, grad_fn=<SqueezeBackward1>)

In [4]:
B, N, d = 10, 100, 37

W = torch.empty(d, 64)
S = torch.empty(B, N, N)
mask = torch.zeros(B, N)
H = torch.empty(B, N, d)

(S @ H).shape, S.shape, H.shape, mask.unsqueeze(-1).shape

H.sum(1).shape

torch.Size([10, 37])

In [ ]:
import torch
from torch import Tensor

class Data:
    def __init__(self, x, edge_index, edge_attr=None, y=None):
        assert edge_index.dtype == torch.long and edge_index.shape[0] == 2
        self.x, self.edge_index, self.edge_attr, self.y = x, edge_index, edge_attr, y

    @property
    def num_nodes(self): return self.x.shape[0]

    @property
    def num_edges(self): return self.edge_index.shape[1]
    

def Batch(Data):
    def __init__(self, x, edge_index, batch, num_graphs, edge_attr=None, y=None):
        super.__init__(x, edge_index, edge_attr, y)
        self.batch, self.num_graphs = batch, num_graphs

    def from_data_list(data_list):
        xs, eis, eas, ys, bvec, off = [], [], [], [], [], 0.0
        for i, d in enumerate(data_list):
            xs.append(d.x)
            eis.append(d.edge_index + off)
            if d.edge_attr is not None: eas.append(d.edge_attr)
            if ys is not None:
                ys.append(d.y.reshape(1) if d.y.dim() == 0 else d.y)
            bvec.append(torch.full((d.num_nodes, ), i, dtype=torch.long))
            off += d.num_nodes
        return Batch(
            x=torch.cat(xs), edge_index=torch.cat(eis, dim=1),
            batch=torch.cat(bvec), num_graphs=len(data_list),
            edge_attr=torch.cat(eas) if eas else None,
            y=torch.cat(ys) if ys else None
        )

def add_self_loops(edge_index, num_nodes):
    loops = torch.arange(num_nodes, dtype=torch.long, device=edge_index.device)
    return torch.cat([edge_index, loops.repeat(2, 1)], dim=1)

def degree(index, num_nodes, dtype):
    return torch.zeros(num_nodes, dtype=dtype, device=index.device).index_add_(
        0, index, torch.ones(index.numel(), dtype=dtype, device=index.device)
    )

def scatter_sum(src, index, dim_size):
    return torch.zeros(src, dim_size.shape[-1], dtype=src.dtype, device=src.device).index_add_(
        0, index, src
    )

def global_mean_pool(x, batch, num_graphs):
    s = scatter_sum(x, batch, num_graphs)
    n = degree(batch, num_graphs, x.dtype).clamp(min=1).unsqueeze(-1)
    return s / n

def global_add_pool(x, batch, num_graphs):
    return scatter_sum(x, batch, num_graphs)

class MessagePassing(nn.Module):
    """edge_index[0] = src[j], edge_index[1] = dst[i];
    message flow j -> i"""

    def __init__(self, aggr='add'):
        super().__init__()
        assert aggr in ('add', 'mean', 'max')
        self.aggr = aggr

    def propagate(self, edge_index, x, num_nodes=None, **kwargs):
        src, dst = edge_index
        n = x.shape[0] if num_nodes is None else num_nodes
        m = self.message(x[src], x[dst], index=dst, **kwargs)
        out = self.aggregate(m, dst, n)
        return self.update(out, **kwargs)

    def message(self, x_j, **kwargs):
        return x_j
    
    def aggregate(self, inputs, index, dim_size):
        if self.aggr == 'add':
            return scatter_sum(inputs, index, dim_size)
        if self.aggr == 'mean':
            return global_mean_pool(inputs, index, dim_size)
        
        out = torch.full(dim_size, inputs.shape[-1], float('-inf'), 
                         device=inputs.device, dtype=inputs.dtype)
        out.index_reduce_(0, index, inputs, 'amax', include_self=True)
        return out.masked_fill(out == float('inf'), 0.0)
    
    def update(self, inputs, **kwargs):
        return inputs
    
message_passer = MessagePassing('add')


In [33]:
graph = mol_to_graph(smiles)

data = Data(graph['X'], graph['E'])

AssertionError: 

In [37]:
graph.keys()

dict_keys(['X', 'A', 'E', 'n', 'smiles'])